# Pipeline Tải File Nguồn arXiv và Đo Lường Hiệu Suất (Milestone 1)

Notebook này phục vụ việc cào và tải file nguồn LaTeX (`.tar.gz` hoặc `.gz`) từ **arXiv** theo khoảng mã bài báo (`start_id` đến `end_id`), hỗ trợ thực thi mượt mà trên môi trường **Google Colab** hoặc máy cục bộ.

### Luồng hoạt động chính:
1. **Cấu hình tham số**: Người dùng gán giá trị cho `start_id` và `end_id`.
2. **Tải file nguồn**: Tải trực tiếp từ mirror `https://export.arxiv.org/src/{arxiv_id}`.
3. **Tuân thủ giới hạn**: Sử dụng `utils.RateLimiter` duy trì khoảng cách tối thiểu **3.0 giây** giữa các yêu cầu.
4. **Giải nén & Lọc ảnh**: Giải nén kho lưu trữ TeX, tự động xóa toàn bộ file ảnh/hình nhị phân (`.png`, `.jpg`, `.pdf`, `.eps`, `.ps`, `.svg`,...) để tối ưu hóa bộ nhớ theo hướng dẫn trong `crawling.md`.
5. **Lưu trữ dữ liệu mã nguồn**: Lưu vào thư mục `data/raw/<clean_arxiv_id>` theo cấu trúc quy định.
6. **Đo lường & Xuất thống kê**: Thu thập toàn bộ thuộc tính theo đúng cấu trúc **Bảng 2: `m1_download_stats`** trong `table_description.md` và lưu ra `data/csv/m1_download_stats.csv`.

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive/')
os.chdir('/content/drive/MyDrive/Colab_Notebooks/arXiv-Project')
# liên kết với drive

Mounted at /content/drive/


## 1. Thiết lập môi trường và cấu hình đường dẫn

In [3]:
# Cài đặt các gói phụ thuộc nếu chạy trên Google Colab
import sys
from pathlib import Path

# Kiểm tra nếu đang chạy trên Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Đang chạy trên Google Colab environment.")
    # Cài đặt các thư viện cần thiết nếu chưa có
    !pip install -q requests pandas pyyaml tqdm

# Tự động xác định PROJECT_ROOT để import module đúng dù notebook chạy ở đâu
current_path = Path.cwd().resolve()
if (current_path / "src").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "src").exists():
    PROJECT_ROOT = current_path.parent
elif (current_path.parent.parent / "src").exists():
    PROJECT_ROOT = current_path.parent.parent
else:
    PROJECT_ROOT = current_path

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Thư mục gốc của dự án (PROJECT_ROOT): {PROJECT_ROOT}")

Đang chạy trên Google Colab environment.
Thư mục gốc của dự án (PROJECT_ROOT): /content/drive/MyDrive/Colab_Notebooks/arXiv-Project


In [4]:
import pandas as pd
from src import config, utils
from src.download_module.downloader import (
    DownloadStats,
    download_single_paper,
    download_papers_range,
    parse_arxiv_range,
    download_papers_range,
)

# Thiết lập thư mục lưu trữ data/raw và data/csv theo quy định
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CSV_DIR = PROJECT_ROOT / "data" / "csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

print(f"Thư mục lưu mã nguồn raw: {RAW_DIR}")
print(f"Thư mục lưu thống kê CSV : {CSV_DIR}")

Thư mục lưu mã nguồn raw: /content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/raw
Thư mục lưu thống kê CSV : /content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/csv


## 2. Cấu hình khoảng mã bài báo (`start_id` và `end_id`)

Người dùng có thể gán giá trị trực tiếp cho `start_id` và `end_id` dưới định dạng chuỗi arXiv chuẩn (ví dụ `"2504.12523"`) hoặc số nguyên kèm theo tham số `month`.

In [10]:
# =============================================================================
# NGƯỜI DÙNG GÁN GIÁ TRỊ CHO start_id VÀ end_id TẠI ĐÂY
# =============================================================================
start_id = "2501.07642"
end_id = "2501.07825"

# Nếu gán số nguyên, chỉ định thêm tháng (Ví dụ: start_id=1, end_id=10, month="2504")
month = None

# Khởi tạo danh sách các ID trong dải
target_ids = parse_arxiv_range(start_id, end_id, month=month)
print(f"Tổng số bài báo trong khoảng cần tải: {len(target_ids)} bài")
print(f"Mã bài báo bắt đầu : {target_ids[0]}")
print(f"Mã bài báo kết thúc: {target_ids[-1]}")
print("Danh sách mẫu:", target_ids[:5])

Tổng số bài báo trong khoảng cần tải: 184 bài
Mã bài báo bắt đầu : 2501.07642
Mã bài báo kết thúc: 2501.07825
Danh sách mẫu: ['2501.07642', '2501.07643', '2501.07644', '2501.07645', '2501.07646']


## 3. Tiến hành tải, đo lường hiệu suất và lọc bỏ dữ liệu ảnh

Hàm `download_papers_range` sẽ:
- Tuân thủ khoảng thời gian chờ tối thiểu 3 giây (`config.ARXIV_MIN_INTERVAL`).
- Tải file nén mã nguồn từ mirror `export.arxiv.org`.
- Nhận diện bài báo chỉ có PDF (`pdf_only`), bỏ qua ảnh nhị phân.
- Giải nén và chỉ giữ các tệp .tex/.bib.
- Đo lường kích thước raw, cleaned, thời gian tải của từng bài báo.
- Mỗi request lỗi tạm thời được gọi tối đa 5 lần tổng cộng; hết lượt ghi lỗi rồi chuyển sang bài tiếp theo.
- `resume=True` bỏ qua source hợp lệ; lỗi tạm thời được thử lại ở lần chạy sau khi hết thời gian chờ (60 giây, tăng đôi, tối đa 1 giờ).
- Dùng `retry_errors=True` sau khi mạng ổn định hoặc đã sửa key/quyền; bật `retry_not_found=True` để kiểm tra lại ID từng trả HTTP 404.
- Tiến độ nằm trong `data/csv/m1_download_stats.ledger.jsonl`; CSV Bảng 2 vẫn giữ nguyên schema.

In [11]:
print(f"Bắt đầu pipeline tải file nguồn arXiv (giới hạn delay >= {config.ARXIV_MIN_INTERVAL}s)...")

# Sử dụng hàm đo lường tài nguyên từ src.utils
with utils.measure("arXiv Pipeline Download & Pruning", disk_path=RAW_DIR) as perf:
    df_stats = download_papers_range(
        start_id=start_id,
        end_id=end_id,
        month=month,
        raw_dir=RAW_DIR,
        csv_dir=CSV_DIR,
        delay=config.ARXIV_MIN_INTERVAL,  # Đảm bảo >= 3.0s theo crawling.md
        resume=True,                      # Bỏ qua source hợp lệ, tiếp tục ID còn dang dở
        retry_errors=False,               # True: thử ngay lỗi sau khi đã sửa nguyên nhân
        retry_not_found=False,            # True: kiểm tra lại các ID trả HTTP 404
        retry_partial=False,              # Cờ dùng chung; download không có trạng thái partial
        show_progress=True,
    )
print("Trạng thái tải theo ID:")
print(df_stats["download_status"].value_counts())

print("\n" + "=" * 60)
print("TỔNG KẾT TÀI NGUYÊN HỆ THỐNG:")
print(f"- Tổng thời gian thực thi: {perf.seconds:.2f} giây")
print(f"- Peak RAM sử dụng       : {perf.peak_rss_mb:.1f} MB")
print(f"- Tổng dung lượng data/raw: {perf.disk_mb:.2f} MB")
print("=" * 60)

Bắt đầu pipeline tải file nguồn arXiv (giới hạn delay >= 3.0s)...



TỔNG KẾT TÀI NGUYÊN HỆ THỐNG:
- Tổng thời gian thực thi: 12.55 giây
- Peak RAM sử dụng       : 272.5 MB
- Tổng dung lượng data/raw: 73.62 MB


## 4. Kiểm tra và xuất bảng dữ liệu thống kê (Bảng 2: `m1_download_stats`)

Bảng dữ liệu theo đúng mô tả cấu trúc trong `table_description.md` gồm các cột:
- `arxiv_id`: Mã bài báo (Primary Key).
- `download_status`: Trạng thái tải (`success`, `failed`, `pdf_only`).
- `has_tex`: Có mã nguồn TeX hay không (`True`/`False`).
- `raw_size_bytes`: Kích thước file nén trước khi lọc ảnh.
- `cleaned_size_bytes`: Kích thước sau khi giải nén và xóa toàn bộ file ảnh/hình nhị phân.
- `download_time_sec`: Thời gian tải và trích xuất nguồn của bài báo.

In [13]:
csv_file = CSV_DIR / "m1_download_stats.csv"
print(f"Dữ liệu bảng thống kê đã được lưu tại: {csv_file}\n")

df_result = pd.read_csv(csv_file)

# Hiển thị các bản ghi đầu tiên
from IPython.display import display
display(df_result.head(50))

# Phân tích số liệu thống kê thu hoạch được
total_count = len(df_result)
success_count = (df_result["download_status"] == "success").sum()
pdf_only_count = (df_result["download_status"] == "pdf_only").sum()
failed_count = (df_result["download_status"] == "failed").sum()

raw_mb = df_result["raw_size_bytes"].sum() / (1024 * 1024)
clean_mb = df_result["cleaned_size_bytes"].sum() / (1024 * 1024)
saved_mb = raw_mb - clean_mb
pct_saved = (saved_mb / raw_mb * 100) if raw_mb > 0 else 0
avg_sec = df_result["download_time_sec"].mean() if total_count > 0 else 0

print("=" * 60)
print("BÁO CÁO KẾT QUẢ THỐNG KÊ (BẢNG 2):")
print(f"- Tổng số bài báo: {total_count}")
print(f"- Số bài tải TeX thành công: {success_count} ({success_count/total_count*100:.1f}%)" if total_count else "")
print(f"- Số bài dạng PDF-only     : {pdf_only_count}")
print(f"- Số bài tải thất bại      : {failed_count}")
print(f"- Dung lượng file nén gốc  : {raw_mb:.2f} MB")
print(f"- Dung lượng sau lọc bỏ ảnh: {clean_mb:.2f} MB")
print(f"- Tiết kiệm bộ nhớ         : {saved_mb:.2f} MB ({pct_saved:.1f}%)")
print(f"- Thời gian trung bình/bài : {avg_sec:.2f} giây")
print("=" * 60)

Dữ liệu bảng thống kê đã được lưu tại: /content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/csv/m1_download_stats.csv



,arxiv_id,download_status,has_tex,raw_size_bytes,cleaned_size_bytes,download_time_sec
0,2501.07642,success,True,1023756,96005,0.7447
1,2501.07643,success,True,83737,118345,0.2896
2,2501.07644,success,True,181156,161311,0.1766
3,2501.07645,success,True,5188,13774,0.0892
4,2501.07646,success,True,288221,317280,0.3774
5,2501.07647,success,True,11119709,160702,1.0986
6,2501.07648,success,True,393545,56970,0.0915
7,2501.07649,success,True,88600,219880,0.3355
8,2501.07650,success,True,25337989,183802,1.6130
9,2501.07651,success,True,23924928,428057,1.0428


BÁO CÁO KẾT QUẢ THỐNG KÊ (BẢNG 2):
- Tổng số bài báo: 184
- Số bài tải TeX thành công: 145 (78.8%)
- Số bài dạng PDF-only     : 12
- Số bài tải thất bại      : 27
- Dung lượng file nén gốc  : 506.66 MB
- Dung lượng sau lọc bỏ ảnh: 73.62 MB
- Tiết kiệm bộ nhớ         : 433.04 MB (85.5%)
- Thời gian trung bình/bài : 2.80 giây


## 5. Kiểm tra cấu trúc thư mục lưu trữ tại `data/raw`

In [14]:
print("Kiểm tra một số thư mục bài báo đã lưu trong data/raw:")
for folder in sorted(RAW_DIR.iterdir())[:5]:
    if folder.is_dir():
        files = list(folder.glob("*"))
        tex_files = [f.name for f in files if f.suffix.lower() == ".tex"]
        bib_files = [f.name for f in files if f.suffix.lower() == ".bib"]
        img_files = [f.name for f in files if f.suffix.lower() in [".png", ".jpg", ".pdf", ".eps"]]
        print(f"- {folder.name}:")
        print(f"    + Tổng số files: {len(files)}")
        print(f"    + File TeX     : {tex_files[:3]}")
        print(f"    + File Bib     : {bib_files[:3]}")
        print(f"    + File ảnh     : {img_files} (phải rỗng do đã lọc bỏ)")

Kiểm tra một số thư mục bài báo đã lưu trong data/raw:
- 2501_07642:
    + Tổng số files: 10
    + File TeX     : ['bench_macros.tex', 'ms.tex', 'review_macros.tex']
    + File Bib     : ['mybib.bib']
    + File ảnh     : [] (phải rỗng do đã lọc bỏ)
- 2501_07643:
    + Tổng số files: 4
    + File TeX     : ['smearedlike.tex']
    + File Bib     : []
    + File ảnh     : [] (phải rỗng do đã lọc bỏ)
- 2501_07644:
    + Tổng số files: 3
    + File TeX     : ['Revised_Journal_Version.tex']
    + File Bib     : ['biblio.bib']
    + File ảnh     : [] (phải rỗng do đã lọc bỏ)
- 2501_07645:
    + Tổng số files: 3
    + File TeX     : ['hallstudentr1.tex']
    + File Bib     : []
    + File ảnh     : [] (phải rỗng do đã lọc bỏ)
- 2501_07646:
    + Tổng số files: 15
    + File TeX     : ['first_iteration.tex', 'flushleft_S1_children.tex', 'flushleft_S1_process.tex']
    + File Bib     : ['main.bib']
    + File ảnh     : [] (phải rỗng do đã lọc bỏ)
